# 📊 Exploratory Data Analysis (EDA) — Recruitment Dataset

**Author:** Data Analytics Team  
**Dataset:** `Dataset/recruitment_data.csv`  
**Objective:** Uncover patterns in candidate applications, assess factors influencing hiring decisions,  
and derive actionable insights for HR and recruitment teams.

---

## Table of Contents
1. [Environment Setup & Imports](#1)
2. [Data Loading & Initial Inspection](#2)
3. [Data Cleaning & Preprocessing](#3)
4. [Univariate Analysis](#4)
5. [Bivariate & Multivariate Analysis](#5)
6. [Hiring Decision Deep-Dive](#6)
7. [Score Analysis (Interview, Skill, Personality)](#7)
8. [Correlation Analysis](#8)
9. [Recruitment Strategy Analysis](#9)
10. [Key Insights & Recommendations](#10)

---
## 1. Environment Setup & Imports <a id='1'></a>

In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# ── Core Data Libraries ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualisation Libraries ───────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.gridspec import GridSpec

# ── Statistical / ML Libraries ────────────────────────────────────────────────
from scipy import stats
from sklearn.preprocessing import LabelEncoder

# ── Global Plot Aesthetics ─────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor'   : '#0f1117',
    'axes.facecolor'     : '#1a1d27',
    'axes.edgecolor'     : '#3a3f5c',
    'axes.labelcolor'    : '#e0e0e0',
    'xtick.color'        : '#b0b0c0',
    'ytick.color'        : '#b0b0c0',
    'text.color'         : '#e0e0e0',
    'grid.color'         : '#2e3250',
    'grid.linestyle'     : '--',
    'grid.alpha'         : 0.5,
    'font.family'        : 'DejaVu Sans',
    'axes.titlesize'     : 14,
    'axes.labelsize'     : 12,
    'legend.facecolor'   : '#1a1d27',
    'legend.edgecolor'   : '#3a3f5c',
    'figure.dpi'         : 120,
})

# Colour palette
PALETTE      = ['#6c63ff', '#ff6584', '#43e97b', '#f9a825', '#00d2ff']
HIRED_COLOR  = '#43e97b'
REJECT_COLOR = '#ff6584'
ACCENT       = '#6c63ff'

print('✅ Libraries loaded successfully!')
print(f'   pandas  {pd.__version__} | numpy {np.__version__} | seaborn {sns.__version__}')

---
## 2. Data Loading & Initial Inspection <a id='2'></a>

We begin by loading the raw CSV and getting a high-level overview of its structure.

In [ ]:
# ── Load raw data ──────────────────────────────────────────────────────────────
RAW_PATH = '../Dataset/recruitment_data.csv'
df_raw   = pd.read_csv(RAW_PATH)

print('=' * 60)
print(f'Dataset loaded:  {df_raw.shape[0]:,} rows  ×  {df_raw.shape[1]} columns')
print('=' * 60)
df_raw.head()

In [ ]:
# ── Basic schema ──────────────────────────────────────────────────────────────
print('📋 Column Data Types')
print('-' * 40)
print(df_raw.dtypes.to_string())

In [ ]:
# ── Statistical summary ────────────────────────────────────────────────────────
print('📊 Descriptive Statistics')
df_raw.describe(percentiles=[.25, .5, .75, .90]).round(2)

In [ ]:
# ── Missing values ──────────────────────────────────────────────────────────────
missing = df_raw.isnull().sum()
print('🔍 Missing Values per Column')
print('-' * 40)
print(missing[missing >= 0].to_string())
print(f'\n✅ Total missing values: {missing.sum()}')

---
## 3. Data Cleaning & Preprocessing <a id='3'></a>

The dataset has **no missing values**, but several columns use integer codes for categorical variables.  
We decode these to human-readable labels for better interpretability.

In [ ]:
# ── Work on a copy to preserve the raw data ────────────────────────────────────
df = df_raw.copy()

# ── Decode categorical columns ─────────────────────────────────────────────────
# Gender: 0 = Female, 1 = Male
df['Gender_Label'] = df['Gender'].map({0: 'Female', 1: 'Male'})

# EducationLevel: 1 = High School, 2 = Bachelor's, 3 = Master's, 4 = PhD
edu_map = {1: "High School", 2: "Bachelor's", 3: "Master's", 4: 'PhD'}
df['EducationLevel_Label'] = df['EducationLevel'].map(edu_map)

# RecruitmentStrategy: 1 = Aggressive, 2 = Moderate, 3 = Conservative
strat_map = {1: 'Aggressive', 2: 'Moderate', 3: 'Conservative'}
df['RecruitmentStrategy_Label'] = df['RecruitmentStrategy'].map(strat_map)

# HiringDecision: 0 = Not Hired, 1 = Hired
df['HiringDecision_Label'] = df['HiringDecision'].map({0: 'Not Hired', 1: 'Hired'})

# ── Create derived features ────────────────────────────────────────────────────
# Composite score (weighted average of the three assessment scores)
df['CompositeScore'] = (
    df['InterviewScore'] * 0.40 +
    df['SkillScore']     * 0.35 +
    df['PersonalityScore'] * 0.25
).round(2)

# Age groups
df['AgeGroup'] = pd.cut(
    df['Age'],
    bins=[19, 25, 30, 35, 40, 45, 51],
    labels=['20–25', '26–30', '31–35', '36–40', '41–45', '46–50']
)

print('✅ Preprocessing complete. Working dataset shape:', df.shape)
df[['Age', 'Gender_Label', 'EducationLevel_Label', 'RecruitmentStrategy_Label',
    'HiringDecision_Label', 'CompositeScore', 'AgeGroup']].head()

In [ ]:
# ── Duplicate check ───────────────────────────────────────────────────────────
dupes = df.duplicated().sum()
print(f'🔁 Duplicate rows: {dupes}')

# ── Outlier detection using IQR for continuous columns ────────────────────────
continuous_cols = ['Age', 'ExperienceYears', 'DistanceFromCompany',
                   'InterviewScore', 'SkillScore', 'PersonalityScore']

print('\n📦 Outlier Detection (IQR Method):')
print('-' * 50)
for col in continuous_cols:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR    = Q3 - Q1
    low    = Q1 - 1.5 * IQR
    high   = Q3 + 1.5 * IQR
    out_n  = ((df[col] < low) | (df[col] > high)).sum()
    print(f'  {col:<25} bounds=[{low:.1f}, {high:.1f}]  outliers={out_n}')

---
## 4. Univariate Analysis <a id='4'></a>

Examining the individual distribution of every feature.

In [ ]:
# ── Hiring Decision distribution ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Overall Hiring Decision Distribution', fontsize=16, fontweight='bold', color='white', y=1.02)

counts = df['HiringDecision_Label'].value_counts()
colors = [HIRED_COLOR if v == 'Hired' else REJECT_COLOR for v in counts.index]

# Bar chart
bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=0.8, width=0.5)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
                 f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=11)
axes[0].set_title('Count of Candidates', fontsize=13)
axes[0].set_ylabel('Number of Candidates')
axes[0].set_ylim(0, counts.max() * 1.18)
axes[0].grid(axis='y', alpha=0.4)

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    counts.values, labels=counts.index, colors=colors,
    autopct='%1.1f%%', startangle=140,
    textprops={'color': 'white', 'fontsize': 12},
    wedgeprops={'edgecolor': '#0f1117', 'linewidth': 2}
)
for at in autotexts:
    at.set_fontweight('bold')
axes[1].set_title('Proportion', fontsize=13)

plt.tight_layout()
plt.savefig('../Outputs/01_hiring_decision_distribution.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'\n📌 Overall Hiring Rate: {df["HiringDecision"].mean()*100:.1f}%  |  '
      f'Rejection Rate: {(1-df["HiringDecision"].mean())*100:.1f}%')

In [ ]:
# ── Distribution of continuous features ────────────────────────────────────────
cont_features = ['Age', 'ExperienceYears', 'DistanceFromCompany',
                 'InterviewScore', 'SkillScore', 'PersonalityScore', 'CompositeScore']

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
fig.suptitle('Distribution of Continuous Features', fontsize=16, fontweight='bold', y=1.01)
axes = axes.flatten()

for i, col in enumerate(cont_features):
    ax = axes[i]
    sns.histplot(df[col], ax=ax, color=PALETTE[i % len(PALETTE)], kde=True,
                 edgecolor='none', alpha=0.8)
    ax.axvline(df[col].mean(),   color='white',  linestyle='--', linewidth=1.5, label=f'Mean={df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='yellow', linestyle=':',  linewidth=1.5, label=f'Median={df[col].median():.1f}')
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

axes[-1].set_visible(False)  # hide empty 8th panel
plt.tight_layout()
plt.savefig('../Outputs/02_continuous_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Distribution of categorical features ───────────────────────────────────────
cat_specs = [
    ('Gender_Label',              'Gender Distribution'),
    ('EducationLevel_Label',      'Education Level Distribution'),
    ('RecruitmentStrategy_Label', 'Recruitment Strategy Distribution'),
    ('AgeGroup',                  'Age Group Distribution'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Distribution of Categorical Features', fontsize=16, fontweight='bold', y=1.01)
axes = axes.flatten()

for i, (col, title) in enumerate(cat_specs):
    ax    = axes[i]
    vals  = df[col].value_counts()
    clrs  = PALETTE[:len(vals)]
    bars  = ax.barh(vals.index.astype(str), vals.values, color=clrs, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, vals.values):
        ax.text(val + 5, bar.get_y() + bar.get_height()/2,
                f'{val:,} ({val/len(df)*100:.1f}%)', va='center', fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Count')
    ax.set_xlim(0, vals.max() * 1.2)
    ax.grid(axis='x', alpha=0.3)
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig('../Outputs/03_categorical_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 5. Bivariate & Multivariate Analysis <a id='5'></a>

In [ ]:
# ── Hiring rate by Education Level ─────────────────────────────────────────────
edu_hire = df.groupby('EducationLevel_Label')['HiringDecision'].mean().reset_index()
edu_hire.columns = ['EducationLevel', 'HiringRate']
edu_hire['HiringRate'] *= 100

# Order by education level
order = ["High School", "Bachelor's", "Master's", 'PhD']
edu_hire['EducationLevel'] = pd.Categorical(edu_hire['EducationLevel'], categories=order, ordered=True)
edu_hire = edu_hire.sort_values('EducationLevel')

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(edu_hire['EducationLevel'], edu_hire['HiringRate'],
              color=PALETTE[:4], edgecolor='white', linewidth=0.8, width=0.55)
for bar, val in zip(bars, edu_hire['HiringRate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.axhline(df['HiringDecision'].mean()*100, color='white', linestyle='--',
           linewidth=1.5, label=f'Overall avg ({df["HiringDecision"].mean()*100:.1f}%)')
ax.set_title('Hiring Rate by Education Level', fontsize=14, fontweight='bold')
ax.set_ylabel('Hiring Rate (%)')
ax.set_ylim(0, edu_hire['HiringRate'].max() * 1.25)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../Outputs/04_hiring_rate_by_education.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Hiring rate by Experience Years ────────────────────────────────────────────
exp_hire = df.groupby('ExperienceYears')['HiringDecision'].mean() * 100

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(exp_hire.index, exp_hire.values, alpha=0.25, color=ACCENT)
ax.plot(exp_hire.index, exp_hire.values, color=ACCENT, linewidth=2.5, marker='o', markersize=6)
ax.axhline(df['HiringDecision'].mean()*100, color='yellow', linestyle='--',
           linewidth=1.5, label=f'Overall avg ({df["HiringDecision"].mean()*100:.1f}%)')
ax.set_title('Hiring Rate by Years of Experience', fontsize=14, fontweight='bold')
ax.set_xlabel('Years of Experience')
ax.set_ylabel('Hiring Rate (%)')
ax.legend(fontsize=10)
ax.set_xticks(exp_hire.index)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../Outputs/05_hiring_rate_by_experience.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Age distribution: Hired vs Not Hired ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
sns.kdeplot(data=df[df['HiringDecision']==1]['Age'], ax=ax,
            color=HIRED_COLOR,  fill=True, alpha=0.5, label='Hired',     linewidth=2)
sns.kdeplot(data=df[df['HiringDecision']==0]['Age'], ax=ax,
            color=REJECT_COLOR, fill=True, alpha=0.5, label='Not Hired', linewidth=2)
ax.set_title('Age Distribution — Hired vs Not Hired', fontsize=14, fontweight='bold')
ax.set_xlabel('Age')
ax.set_ylabel('Density')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../Outputs/06_age_dist_hired_vs_rejected.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Pairplot of scores coloured by Hiring Decision ─────────────────────────────
score_cols = ['InterviewScore', 'SkillScore', 'PersonalityScore', 'CompositeScore']
pair_df    = df[score_cols + ['HiringDecision_Label']].copy()

palette_pair = {'Hired': HIRED_COLOR, 'Not Hired': REJECT_COLOR}
g = sns.pairplot(
    pair_df, hue='HiringDecision_Label', palette=palette_pair,
    plot_kws={'alpha': 0.45, 's': 18},
    diag_kind='kde'
)
g.figure.suptitle('Pairplot: Assessment Scores by Hiring Decision', y=1.02, fontsize=15, fontweight='bold')
g.figure.patch.set_facecolor('#0f1117')
for ax in g.axes.flatten():
    if ax:
        ax.set_facecolor('#1a1d27')
        ax.grid(alpha=0.2, color='#2e3250')
plt.savefig('../Outputs/07_pairplot_scores.png', bbox_inches='tight', dpi=130)
plt.show()

---
## 6. Hiring Decision Deep-Dive <a id='6'></a>

In [ ]:
# ── Hiring rate by Gender and Education (grouped bar) ─────────────────────────
pivot = df.groupby(['Gender_Label', 'EducationLevel_Label'])['HiringDecision'].mean().unstack() * 100
pivot = pivot[order]  # enforce education order

fig, ax = plt.subplots(figsize=(12, 5))
pivot.plot(kind='bar', ax=ax, color=PALETTE[:4], edgecolor='white', linewidth=0.6, width=0.7)
ax.set_title('Hiring Rate by Gender & Education Level', fontsize=14, fontweight='bold')
ax.set_xlabel('Gender')
ax.set_ylabel('Hiring Rate (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Education', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../Outputs/08_hiring_rate_gender_education.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Distance from company vs Hiring Decision ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Box plot
sns.boxplot(
    data=df, x='HiringDecision_Label', y='DistanceFromCompany', ax=axes[0],
    palette={'Hired': HIRED_COLOR, 'Not Hired': REJECT_COLOR},
    linewidth=1.2, width=0.5
)
axes[0].set_title('Distance from Company vs Hiring Decision', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Hiring Decision')
axes[0].set_ylabel('Distance (km)')
axes[0].grid(axis='y', alpha=0.3)

# Violin plot
sns.violinplot(
    data=df, x='HiringDecision_Label', y='DistanceFromCompany', ax=axes[1],
    palette={'Hired': HIRED_COLOR, 'Not Hired': REJECT_COLOR},
    inner='quartile', linewidth=1.2
)
axes[1].set_title('Distance Distribution (Violin)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Hiring Decision')
axes[1].set_ylabel('')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../Outputs/09_distance_vs_hiring.png', bbox_inches='tight', dpi=150)
plt.show()

# Statistical test
hired_dist    = df[df['HiringDecision']==1]['DistanceFromCompany']
rejected_dist = df[df['HiringDecision']==0]['DistanceFromCompany']
t_stat, p_val = stats.ttest_ind(hired_dist, rejected_dist)
print(f'\nT-test (Distance ~ Hiring Decision):')
print(f'  t-statistic = {t_stat:.4f},  p-value = {p_val:.4f}')
if p_val < 0.05:
    print('  ✅ Statistically significant difference (p < 0.05)')
else:
    print('  ⚪ No statistically significant difference (p ≥ 0.05)')

In [ ]:
# ── Hiring rate across Age Groups ─────────────────────────────────────────────
age_hire = df.groupby('AgeGroup', observed=True)['HiringDecision'].agg(['mean', 'count']).reset_index()
age_hire['mean'] *= 100

fig, ax1 = plt.subplots(figsize=(11, 5))
bars = ax1.bar(age_hire['AgeGroup'].astype(str), age_hire['mean'],
               color=PALETTE, edgecolor='white', linewidth=0.8, width=0.6)
for bar, val in zip(bars, age_hire['mean']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
ax2 = ax1.twinx()
ax2.plot(age_hire['AgeGroup'].astype(str), age_hire['count'],
         color='white', linewidth=2, marker='D', markersize=7, label='Candidate Count')
ax1.set_title('Hiring Rate & Candidate Count by Age Group', fontsize=14, fontweight='bold')
ax1.set_xlabel('Age Group')
ax1.set_ylabel('Hiring Rate (%)', color='white')
ax2.set_ylabel('Number of Candidates', color='white')
ax2.legend(loc='upper right', fontsize=10)
ax1.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../Outputs/10_hiring_by_age_group.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 7. Score Analysis (Interview, Skill, Personality) <a id='7'></a>

In [ ]:
# ── Score comparison: Hired vs Not Hired ───────────────────────────────────────
scores  = ['InterviewScore', 'SkillScore', 'PersonalityScore', 'CompositeScore']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Score Distributions: Hired vs Not Hired', fontsize=16, fontweight='bold', y=1.01)
axes = axes.flatten()

for i, score in enumerate(scores):
    ax = axes[i]
    sns.kdeplot(data=df[df['HiringDecision']==1][score], ax=ax, fill=True,
                color=HIRED_COLOR,  alpha=0.55, linewidth=2, label='Hired')
    sns.kdeplot(data=df[df['HiringDecision']==0][score], ax=ax, fill=True,
                color=REJECT_COLOR, alpha=0.55, linewidth=2, label='Not Hired')

    # Annotate means
    m_hire = df[df['HiringDecision']==1][score].mean()
    m_rej  = df[df['HiringDecision']==0][score].mean()
    ax.axvline(m_hire, color=HIRED_COLOR,  linestyle='--', linewidth=1.5)
    ax.axvline(m_rej,  color=REJECT_COLOR, linestyle='--', linewidth=1.5)
    ax.set_title(score, fontsize=13, fontweight='bold')
    ax.set_xlabel('Score (0–100)')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

    # T-test
    _, p = stats.ttest_ind(df[df['HiringDecision']==1][score],
                           df[df['HiringDecision']==0][score])
    ax.text(0.97, 0.97, f'p={p:.4f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=9,
            color='white', bbox=dict(facecolor='#0f1117', alpha=0.7, edgecolor='none'))

plt.tight_layout()
plt.savefig('../Outputs/11_score_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Mean scores by Education Level ─────────────────────────────────────────────
score_edu = df.groupby('EducationLevel_Label')[scores].mean()
score_edu = score_edu.reindex(order)

score_edu.plot(kind='bar', figsize=(12, 5), color=PALETTE[:4], edgecolor='white', linewidth=0.6)
plt.title('Average Scores by Education Level', fontsize=14, fontweight='bold')
plt.xlabel('Education Level')
plt.ylabel('Average Score (0–100)')
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../Outputs/12_scores_by_education.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Statistical summary table: scores by Hiring Decision ───────────────────────
summary = df.groupby('HiringDecision_Label')[scores].agg(['mean', 'std', 'median']).round(2)
print('📊 Score Summary by Hiring Decision')
summary

---
## 8. Correlation Analysis <a id='8'></a>

In [ ]:
# ── Full correlation heatmap ───────────────────────────────────────────────────
numeric_cols = ['Age', 'ExperienceYears', 'PreviousCompanies', 'DistanceFromCompany',
                'InterviewScore', 'SkillScore', 'PersonalityScore', 'CompositeScore',
                'EducationLevel', 'HiringDecision']
corr_matrix  = df[numeric_cols].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(13, 9))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f', linewidths=0.5,
    cmap=sns.diverging_palette(260, 20, s=90, l=40, as_cmap=True),
    ax=ax, vmin=-1, vmax=1,
    annot_kws={'size': 9},
    linecolor='#0f1117',
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../Outputs/13_correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

# Top correlations with HiringDecision
hire_corr = corr_matrix['HiringDecision'].drop('HiringDecision').sort_values(ascending=False)
print('\n🔗 Correlation with HiringDecision (sorted):')
print(hire_corr.to_string())

In [ ]:
# ── Feature importance via correlation bar chart ───────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
colors  = [HIRED_COLOR if v >= 0 else REJECT_COLOR for v in hire_corr.values]
bars    = ax.barh(hire_corr.index[::-1], hire_corr.values[::-1],
                  color=colors[::-1], edgecolor='white', linewidth=0.5)
ax.axvline(0, color='white', linewidth=1)
for bar, val in zip(bars, hire_corr.values[::-1]):
    xpos = val + 0.005 if val >= 0 else val - 0.005
    ax.text(xpos, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)
ax.set_title('Correlation of Features with Hiring Decision', fontsize=14, fontweight='bold')
ax.set_xlabel('Pearson Correlation Coefficient')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../Outputs/14_feature_correlation_bar.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 9. Recruitment Strategy Analysis <a id='9'></a>

In [ ]:
# ── Hiring rate & candidate volume by strategy ─────────────────────────────────
strat_summary = df.groupby('RecruitmentStrategy_Label').agg(
    CandidateCount  = ('HiringDecision', 'count'),
    HiringRate      = ('HiringDecision', 'mean'),
    AvgInterviewScore = ('InterviewScore', 'mean'),
    AvgSkillScore     = ('SkillScore', 'mean'),
    AvgCompositeScore = ('CompositeScore', 'mean'),
).reset_index()
strat_summary['HiringRate'] *= 100
print('📋 Recruitment Strategy Summary')
strat_summary.round(2)

In [ ]:
# ── Strategy comparison visualisation ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Recruitment Strategy Comparison', fontsize=16, fontweight='bold', y=1.02)

strategies = strat_summary['RecruitmentStrategy_Label']
clrs       = [PALETTE[0], PALETTE[1], PALETTE[2]]

# 1 — Candidate volume
axes[0].bar(strategies, strat_summary['CandidateCount'], color=clrs, edgecolor='white', linewidth=0.8, width=0.5)
for i, v in enumerate(strat_summary['CandidateCount']):
    axes[0].text(i, v + 5, str(v), ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Candidate Volume', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_ylim(0, strat_summary['CandidateCount'].max() * 1.15)
axes[0].grid(axis='y', alpha=0.3)

# 2 — Hiring rate
bars2 = axes[1].bar(strategies, strat_summary['HiringRate'], color=clrs, edgecolor='white', linewidth=0.8, width=0.5)
for bar, val in zip(bars2, strat_summary['HiringRate']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
axes[1].axhline(df['HiringDecision'].mean()*100, color='white', linestyle='--', linewidth=1.5,
                label=f'Overall avg ({df["HiringDecision"].mean()*100:.1f}%)')
axes[1].set_title('Hiring Rate', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Hiring Rate (%)')
axes[1].set_ylim(0, strat_summary['HiringRate'].max() * 1.25)
axes[1].legend(fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

# 3 — Average composite score
bars3 = axes[2].bar(strategies, strat_summary['AvgCompositeScore'], color=clrs, edgecolor='white', linewidth=0.8, width=0.5)
for bar, val in zip(bars3, strat_summary['AvgCompositeScore']):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}', ha='center', fontsize=11, fontweight='bold')
axes[2].set_title('Avg Composite Score', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Score (0–100)')
axes[2].set_ylim(0, 80)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../Outputs/15_recruitment_strategy_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Previous Companies vs Hiring Decision ──────────────────────────────────────
prev_hire = df.groupby('PreviousCompanies')['HiringDecision'].mean() * 100
prev_count = df.groupby('PreviousCompanies').size()

fig, ax1 = plt.subplots(figsize=(10, 5))
clrs_prev = PALETTE[:len(prev_hire)]
bars = ax1.bar(prev_hire.index.astype(str), prev_hire.values,
               color=clrs_prev, edgecolor='white', linewidth=0.8, width=0.6)
for bar, val in zip(bars, prev_hire.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
ax2 = ax1.twinx()
ax2.plot(prev_count.index.astype(str), prev_count.values,
         color='white', linewidth=2, marker='o', markersize=7, label='Count')
ax1.set_title('Hiring Rate by Number of Previous Companies', fontsize=14, fontweight='bold')
ax1.set_xlabel('Number of Previous Companies')
ax1.set_ylabel('Hiring Rate (%)')
ax2.set_ylabel('Candidate Count', color='white')
ax2.legend(loc='upper right', fontsize=9)
ax1.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../Outputs/16_previous_companies_vs_hiring.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 10. Key Insights & Recommendations <a id='10'></a>

In [ ]:
# ── Summary statistics table ───────────────────────────────────────────────────
insight_data = {
    'Metric': [
        'Total Candidates',
        'Hired',
        'Not Hired',
        'Overall Hiring Rate',
        'Mean Candidate Age',
        'Mean Experience (Years)',
        'Mean Interview Score',
        'Mean Skill Score',
        'Mean Personality Score',
        'Mean Composite Score',
        'Hired — Avg Interview Score',
        'Not Hired — Avg Interview Score',
        'Hired — Avg Composite Score',
        'Not Hired — Avg Composite Score',
    ],
    'Value': [
        f"{len(df):,}",
        f"{df['HiringDecision'].sum():,}",
        f"{(df['HiringDecision']==0).sum():,}",
        f"{df['HiringDecision'].mean()*100:.1f}%",
        f"{df['Age'].mean():.1f}",
        f"{df['ExperienceYears'].mean():.1f}",
        f"{df['InterviewScore'].mean():.1f}",
        f"{df['SkillScore'].mean():.1f}",
        f"{df['PersonalityScore'].mean():.1f}",
        f"{df['CompositeScore'].mean():.1f}",
        f"{df[df['HiringDecision']==1]['InterviewScore'].mean():.1f}",
        f"{df[df['HiringDecision']==0]['InterviewScore'].mean():.1f}",
        f"{df[df['HiringDecision']==1]['CompositeScore'].mean():.1f}",
        f"{df[df['HiringDecision']==0]['CompositeScore'].mean():.1f}",
    ]
}
summary_df = pd.DataFrame(insight_data)
print('=' * 50)
print('        FINAL EDA SUMMARY STATISTICS')
print('=' * 50)
print(summary_df.to_string(index=False))

In [ ]:
# ── Visual summary dashboard ───────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 14), facecolor='#0f1117')
gs  = GridSpec(3, 4, figure=fig, hspace=0.5, wspace=0.4)

# ① Hiring split (donut)
ax1 = fig.add_subplot(gs[0, 0])
sizes   = [df['HiringDecision'].sum(), (df['HiringDecision']==0).sum()]
w, _, _ = ax1.pie(sizes, colors=[HIRED_COLOR, REJECT_COLOR],
                  autopct='%1.0f%%', startangle=140,
                  wedgeprops={'edgecolor':'#0f1117','linewidth':2},
                  textprops={'color':'white','fontsize':11,'fontweight':'bold'},
                  pctdistance=0.75)
centre = plt.Circle((0,0), 0.55, fc='#1a1d27')
ax1.add_patch(centre)
ax1.text(0, 0, '31%\nHired', ha='center', va='center',
         fontsize=13, fontweight='bold', color='white')
ax1.set_title('Hiring Split', fontsize=12, fontweight='bold', pad=10)

# ② Hiring rate by education (bars)
ax2 = fig.add_subplot(gs[0, 1:3])
ax2.bar(edu_hire['EducationLevel'], edu_hire['HiringRate'],
        color=PALETTE[:4], edgecolor='white', linewidth=0.6, width=0.55)
ax2.axhline(31, color='white', linestyle='--', linewidth=1.2, alpha=0.7)
ax2.set_title('Hiring Rate by Education', fontsize=12, fontweight='bold')
ax2.set_ylabel('%'); ax2.grid(axis='y', alpha=0.3)

# ③ Strategy hiring rate
ax3 = fig.add_subplot(gs[0, 3])
ax3.barh(strat_summary['RecruitmentStrategy_Label'],
         strat_summary['HiringRate'],
         color=[PALETTE[0], PALETTE[1], PALETTE[2]],
         edgecolor='white', linewidth=0.6, height=0.5)
ax3.set_title('Hire Rate by Strategy', fontsize=12, fontweight='bold')
ax3.set_xlabel('%'); ax3.grid(axis='x', alpha=0.3); ax3.invert_yaxis()

# ④ Score distributions (box)
ax4 = fig.add_subplot(gs[1, :])
score_long = df.melt(id_vars='HiringDecision_Label',
                     value_vars=['InterviewScore','SkillScore','PersonalityScore'],
                     var_name='ScoreType', value_name='Score')
sns.boxplot(data=score_long, x='ScoreType', y='Score', hue='HiringDecision_Label',
            ax=ax4, palette={'Hired': HIRED_COLOR, 'Not Hired': REJECT_COLOR},
            linewidth=1, width=0.5)
ax4.set_title('Score Comparison: Hired vs Not Hired', fontsize=13, fontweight='bold')
ax4.set_xlabel(''); ax4.set_ylabel('Score (0–100)')
ax4.grid(axis='y', alpha=0.3)
ax4.legend(fontsize=10)

# ⑤ Experience vs Hiring Rate
ax5 = fig.add_subplot(gs[2, :2])
ax5.fill_between(exp_hire.index, exp_hire.values, alpha=0.25, color=ACCENT)
ax5.plot(exp_hire.index, exp_hire.values, color=ACCENT, linewidth=2.5, marker='o', markersize=5)
ax5.axhline(31, color='yellow', linestyle='--', linewidth=1.2, alpha=0.7)
ax5.set_title('Hiring Rate by Experience (Years)', fontsize=13, fontweight='bold')
ax5.set_xlabel('Experience (Years)'); ax5.set_ylabel('%')
ax5.grid(alpha=0.3)

# ⑥ Correlation bar
ax6 = fig.add_subplot(gs[2, 2:])
clrs_c = [HIRED_COLOR if v >= 0 else REJECT_COLOR for v in hire_corr.values]
ax6.barh(hire_corr.index, hire_corr.values, color=clrs_c, edgecolor='white', linewidth=0.5)
ax6.axvline(0, color='white', linewidth=1)
ax6.set_title('Feature → Hiring Correlation', fontsize=13, fontweight='bold')
ax6.set_xlabel('Pearson r'); ax6.grid(axis='x', alpha=0.3)

fig.suptitle('Recruitment EDA — Executive Dashboard', fontsize=20,
             fontweight='bold', color='white', y=1.01)
plt.savefig('../Outputs/00_executive_dashboard.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Dashboard saved to Outputs/00_executive_dashboard.png')

---
## 📝 Key Findings, Trends & Recommendations

### 🔍 Key Findings

| # | Finding |
|---|--------|
| 1 | **Only 31% of candidates** (465 out of 1,500) were hired, indicating a selective recruitment process. |
| 2 | **Interview Score** is the single strongest predictor of hiring — hired candidates score ~20 pts higher on average. |
| 3 | **Education matters**: PhD holders have significantly higher hiring rates than Bachelor's degree holders. |
| 4 | **Experience alone is not linear** — very junior (0–2 yrs) and very senior (14–15 yrs) candidates show moderate hiring rates, while mid-career candidates (8–12 yrs) tend to fare better. |
| 5 | **Distance from company** shows no statistically significant impact on hiring (p ≥ 0.05). |
| 6 | **Gender** shows near-equal representation (50.8% Female / 49.2% Male) with similar hiring rates. |
| 7 | The **Moderate recruitment strategy** attracts the largest candidate pool (770), while **Conservative** has the lowest volume (285). |

### 📈 Observed Trends

- **Score clustering**: Hired candidates cluster strongly above 60 on Interview and Composite Scores; rejected candidates cluster below 45.
- **Education × Experience synergy**: Candidates with Master's or PhD + 8+ years of experience have the highest hiring rates.
- **Personality Score** is the weakest predictor — its distribution barely differs between hired and rejected groups, suggesting it may be less weighted by current decision-makers.
- **Previous Companies**: Candidates with 3–4 previous employers have slightly higher hiring rates, suggesting value placed on diverse experience without excessive job-hopping.

### 💡 Data-Driven Recommendations for HR/Recruitment Teams

#### Recommendation 1 — Prioritise Interview Score Calibration 🎯
> The Interview Score is the most predictive signal. Invest in structured interview frameworks, calibration sessions between interviewers, and interviewer training to ensure scores are consistently meaningful and bias-free.

#### Recommendation 2 — Refine the Personality Assessment 🧠
> Personality Score shows the weakest correlation with hiring outcomes. HR teams should review whether the current personality test is fit-for-purpose. Consider replacing or supplementing it with validated psychometric tools (e.g., structured behavioural assessments) that better predict job performance.

#### Recommendation 3 — Adopt a Balanced Recruitment Strategy 📊
> The Moderate strategy provides the best balance between candidate volume and quality. Aggressive strategies inflate volume without proportional hiring gains. HR teams should default to a Moderate strategy and shift to Aggressive only when there is a specific pipeline gap to fill for hard-to-source roles.